# Walkthrough — Error-Driven Question Recommender

This notebook is a quick demo of the `edqr` package:

1. Load the trained ADGKT checkpoint.
2. Run **Prediction-Select-Verification (PSV)** on a sample of held-out users:
   mask the future, flag questions the model predicts the student will get
   wrong, verify against the student's real answers.
3. Inspect one student's flagged weak points.

Prerequisites: the XES3G5M dataset under `data/XES3G5M/` (see `data/README.md`)
and a trained checkpoint under `models/` (`python -m edqr.train --model adgkt --data-dir data/XES3G5M`).

In [ ]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

DATA_DIR = Path("data/XES3G5M")   # adjust if the dataset lives elsewhere
MODELS_DIR = Path("models")

from edqr.config import DataConfig, EvalConfig, TrainConfig
from edqr.data import clean_test_sequences, load_question_modules
from edqr.evaluate import load_model, predict_next_probs, prepare_eval_users

## 1. Load model and held-out users

The evaluation users come from the official XES3G5M `test.csv` split — **disjoint from every user the model trained on** (cold start by construction).

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, qid_map, cid_map, train_cfg = load_model(MODELS_DIR, device)

data_cfg = DataConfig()
test_df = prepare_eval_users(DATA_DIR, qid_map, data_cfg)
test_df = test_df.head(100)   # small sample for the demo
print(f"sample: {len(test_df)} users, device={device}")

## 2. Prediction-Select-Verification

For every step *t* of a student's sequence the model sees only interactions
before *t* and scores the next question. `p(correct) <= theta` flags the
question as a **predicted error** — the recommendation. Afterwards we compare
with the student's actual response at *t*.

In [ ]:
eval_cfg = EvalConfig()
theta = 0.1

records = []
for _, row in test_df.iterrows():
    q_seq = str(row["questions"]).split(",")
    c_seq = str(row["concepts"]).split(",")
    r_seq = [int(r) for r in str(row["responses"]).split(",")]
    probs, actual = predict_next_probs(model, q_seq, c_seq, r_seq,
                                       qid_map, cid_map, device,
                                       TrainConfig.max_seq_len, eval_cfg.row_batch_size)
    flags = probs <= theta
    records.append({"user": row["uid"],
                    "positions": len(probs),
                    "flagged": int(flags.sum()),
                    "verified_wrong": int((flags & (actual == 0)).sum()),
                    "natural_wrong": int((actual == 0).sum())})

df = pd.DataFrame(records)
flagged, verified = df["flagged"].sum(), df["verified_wrong"].sum()
positions, natural_wrong = df["positions"].sum(), df["natural_wrong"].sum()
natural_rate = natural_wrong / positions
precision = verified / flagged if flagged else 0

print(f"users evaluated:        {len(df)}")
print(f"flagged questions:      {flagged}")
print(f"verified real errors:   {verified}  (precision {precision:.1%})")
print(f"natural error rate:     {natural_rate:.1%}")
print(f"lift:                   {precision / natural_rate:.1f}x")
print(f"users with >=1 flag:    {(df['flagged'] > 0).mean():.1%}")

## 3. One student's diagnosis

The flagged questions are exactly what the recommender would push to this
student: model says *you will most likely get these wrong*.

In [ ]:
row = test_df.iloc[0]
q_seq = str(row["questions"]).split(",")
c_seq = str(row["concepts"]).split(",")
r_seq = [int(r) for r in str(row["responses"]).split(",")]
probs, actual = predict_next_probs(model, q_seq, c_seq, r_seq,
                                   qid_map, cid_map, device,
                                   TrainConfig.max_seq_len, eval_cfg.row_batch_size)

steps = range(1, len(q_seq))  # approximate alignment for display
flags = np.where(probs <= theta)[0]
print(f"user {row['uid']}: {len(q_seq)} interactions, {len(flags)} flagged
")
print(f"{'step':>5} {'question':>10} {'module':<12} {'p(correct)':>10} {'actual':>7}")
for i in flags[:12]:
    print(f"{i+1:>5} {q_seq[i]:>10} {c_seq[i]:<12} {probs[i]:>10.3f} "
          f"{'WRONG' if actual[i] == 0 else 'right':>7}")

## 4. Full-scale results

The numbers below come from the full run over all evaluation users
(`python -m edqr.evaluate --mode sweep --data-dir data/XES3G5M`),
stored in `results/psv_results.json`.

In [ ]:
with open("results/psv_results.json", encoding="utf-8") as f:
    full = json.load(f)

rows = []
for t, m in full["thresholds"].items():
    rows.append({"theta": float(t), "precision": m["precision"],
                 "coverage": m["user_coverage"], "lift": m["lift_over_natural"],
                 "flags/user": m["avg_flags_per_user"]})
display(pd.DataFrame(rows).set_index("theta"))